In [ ]:
# ── Cell 1: Mount Drive ───────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os

TRAIN_DIR = '/content/drive/MyDrive/Alz/dataset/train'
TEST_DIR  = '/content/drive/MyDrive/Alz/dataset/test'
SAVE_DIR  = '/content/drive/MyDrive/Alz/models'

os.makedirs(SAVE_DIR, exist_ok=True)

print('Drive mounted!')

In [ ]:
# ── Cell 2: Install libraries ─────────────────────────────────

!pip install -q tensorflow scikit-learn matplotlib seaborn joblib Pillow opencv-python

In [ ]:
# ── Cell 3: Define CLASS_NAMES ────────────────────────────────

import os

CLASS_NAMES = sorted([
    f for f in os.listdir(TRAIN_DIR)
    if os.path.isdir(os.path.join(TRAIN_DIR, f))
])

print('Classes found:', CLASS_NAMES)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 4 — Imports & Build Feature Extractors
# Two backbones: MobileNetV2 (lightweight) + ResNet50 (deeper)
# ════════════════════════════════════════════════════════════════

import numpy as np
import os
import cv2
import json
import joblib
from PIL import Image

import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import seaborn as sns

from tensorflow.keras.applications import MobileNetV2, ResNet50
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50   import preprocess_input as resnet_preprocess

from sklearn.svm                    import SVC
from sklearn.ensemble               import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing          import LabelEncoder
from sklearn.metrics                import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight     import compute_class_weight

tf.keras.backend.clear_session()

# ── MobileNetV2 — lightweight backbone (128×128 input) ──────────
MOBILENET_IMG_SIZE = (128, 128)

mobilenet_extractor = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3),
    pooling='avg'          # output shape: (None, 1280)
)
mobilenet_extractor.trainable = False

# ── ResNet50 — deeper backbone (224×224 input) ───────────────────
RESNET_IMG_SIZE = (224, 224)

resnet_extractor = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3),
    pooling='avg'          # output shape: (None, 2048)
)
resnet_extractor.trainable = False

print(f"MobileNetV2 feature vector : {mobilenet_extractor.output_shape[1]}")
print(f"ResNet50    feature vector : {resnet_extractor.output_shape[1]}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 5 — Feature Extraction (MobileNetV2 + ResNet50)
# ════════════════════════════════════════════════════════════════

def extract_features(data_dir, class_names, extractor, preprocess_fn, img_size):
    """
    Extract deep features from all images using a frozen CNN backbone.
    Returns X (feature matrix) and y (string labels).
    """
    X, y = [], []
    for cls in class_names:
        cls_path = os.path.join(data_dir, cls)
        files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  {cls:<25} ({len(files)} images)")
        for fname in files:
            img = Image.open(os.path.join(cls_path, fname)).convert('RGB').resize(img_size)
            arr = preprocess_fn(
                np.expand_dims(np.array(img, dtype=np.float32), axis=0)
            )
            X.append(extractor.predict(arr, verbose=0)[0])
            y.append(cls)
    return np.array(X), np.array(y)


# ── MobileNetV2 features ──────────────────────────────────────────
print("[MobileNetV2] Extracting TRAIN features...")
X_train_mob, y_train = extract_features(
    TRAIN_DIR, CLASS_NAMES, mobilenet_extractor, mobilenet_preprocess, MOBILENET_IMG_SIZE
)
print("\n[MobileNetV2] Extracting TEST features...")
X_test_mob, y_test = extract_features(
    TEST_DIR, CLASS_NAMES, mobilenet_extractor, mobilenet_preprocess, MOBILENET_IMG_SIZE
)

# ── ResNet50 features ─────────────────────────────────────────────
print("\n[ResNet50] Extracting TRAIN features...")
X_train_res, _ = extract_features(
    TRAIN_DIR, CLASS_NAMES, resnet_extractor, resnet_preprocess, RESNET_IMG_SIZE
)
print("\n[ResNet50] Extracting TEST features...")
X_test_res, _ = extract_features(
    TEST_DIR, CLASS_NAMES, resnet_extractor, resnet_preprocess, RESNET_IMG_SIZE
)

print(f"\nMobileNetV2 — Train: {X_train_mob.shape}, Test: {X_test_mob.shape}")
print(f"ResNet50    — Train: {X_train_res.shape}, Test: {X_test_res.shape}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 6 — Label Encoding & Class Weights
# ════════════════════════════════════════════════════════════════

le = LabelEncoder()
le.fit(CLASS_NAMES)

y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

# Compute balanced class weights (cap at 5.0 to avoid extreme penalties)
cw_arr  = compute_class_weight('balanced', classes=np.unique(y_train_enc), y=y_train_enc)
cw_dict = {i: min(w, 5.0) for i, w in enumerate(cw_arr)}

print("Label mapping :", dict(zip(le.classes_, le.transform(le.classes_))))
print("Class weights :", {le.classes_[k]: round(v, 2) for k, v in cw_dict.items()})

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 7 — Train All Classifiers on Both Feature Sets
# Classifiers: SVM · Random Forest · Gradient Boosting
# Backbones  : MobileNetV2 · ResNet50
# ════════════════════════════════════════════════════════════════

def get_classifiers():
    """Return freshly initialised classifiers (called per backbone)."""
    return {
        'SVM': SVC(
            kernel='rbf', C=10, gamma='scale',
            class_weight='balanced', probability=True
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, class_weight='balanced',
            random_state=42, n_jobs=-1
        ),
        'Gradient Boosting': GradientBoostingClassifier(
            n_estimators=100, random_state=42
        )
    }


def train_and_evaluate(X_train, X_test, y_train_enc, y_test_enc, backbone_name):
    """Train all classifiers, evaluate on test set, and return results dict."""
    classifiers = get_classifiers()
    results = {}
    print(f"\n{'='*55}")
    print(f"  Backbone : {backbone_name}")
    print(f"  Features : {X_train.shape[1]}-dimensional vector")
    print(f"{'='*55}")
    for name, clf in classifiers.items():
        print(f"  Training {name}...", end='  ')
        clf.fit(X_train, y_train_enc)
        y_pred = clf.predict(X_test)
        acc    = accuracy_score(y_test_enc, y_pred)
        results[name] = {'clf': clf, 'acc': acc, 'pred': y_pred}
        print(f"Test Accuracy: {acc * 100:.2f}%")
    return results


results_mob = train_and_evaluate(
    X_train_mob, X_test_mob, y_train_enc, y_test_enc, 'MobileNetV2'
)
results_res = train_and_evaluate(
    X_train_res, X_test_res, y_train_enc, y_test_enc, 'ResNet50'
)

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 8 — Evaluation: Classification Reports + Confusion Matrices
# ════════════════════════════════════════════════════════════════

def plot_confusion_matrix(y_true, y_pred, class_names, title, save_path=None):
    """Plot raw-count and normalised confusion matrices side-by-side."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=axes[0])
    axes[0].set_title('Raw Counts')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    axes[0].tick_params(axis='x', rotation=30)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=axes[1])
    axes[1].set_title('Normalised (row = recall)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
    axes[1].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


for backbone, results in [('MobileNetV2', results_mob), ('ResNet50', results_res)]:
    print(f"\n{'='*60}")
    print(f"  {backbone} — Classification Reports")
    print(f"{'='*60}")

    for clf_name, r in results.items():
        print(f"\n[ {backbone} + {clf_name} ]   Accuracy: {r['acc']*100:.2f}%")
        print(classification_report(y_test_enc, r['pred'], target_names=CLASS_NAMES))

    # Confusion matrix for the best-performing (non-overfitting) classifier
    valid    = {k: v for k, v in results.items() if v['acc'] < 1.0} or results
    best_name = max(valid, key=lambda k: valid[k]['acc'])
    plot_confusion_matrix(
        y_test_enc, results[best_name]['pred'], CLASS_NAMES,
        f'Confusion Matrix — {backbone} + {best_name}',
        save_path=f"{SAVE_DIR}/cm_{backbone.lower()}.png"
    )

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 9 — Model Comparison: MobileNetV2 vs ResNet50
# All 6 combinations (2 backbones × 3 classifiers) on one chart
# ════════════════════════════════════════════════════════════════

clf_names  = list(results_mob.keys())
x          = np.arange(len(clf_names))
width      = 0.35

mob_accs = [results_mob[k]['acc'] * 100 for k in clf_names]
res_accs = [results_res[k]['acc'] * 100 for k in clf_names]

fig, ax = plt.subplots(figsize=(12, 6))

bars1 = ax.bar(x - width / 2, mob_accs, width,
               label='MobileNetV2', color='#3498db', edgecolor='white')
bars2 = ax.bar(x + width / 2, res_accs, width,
               label='ResNet50',    color='#e74c3c', edgecolor='white')

# Annotate bars with accuracy values
for bar in list(bars1) + list(bars2):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f'{bar.get_height():.1f}%',
        ha='center', fontsize=9, fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(clf_names, fontsize=11)
ax.set_ylim(0, 110)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title(
    'Backbone Comparison — MobileNetV2 vs ResNet50\n'
    '(SVM · Random Forest · Gradient Boosting)',
    fontsize=13, fontweight='bold'
)
ax.axhline(70, color='gray', linestyle='--', alpha=0.5, label='70% baseline')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/backbone_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f"\n{'Classifier':<22} {'MobileNetV2':>14} {'ResNet50':>12}")
print('-' * 50)
for k in clf_names:
    print(f"{k:<22} {results_mob[k]['acc']*100:>13.2f}% {results_res[k]['acc']*100:>11.2f}%")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 10 — Save All Models & Shared Assets
# ════════════════════════════════════════════════════════════════

def safe_name(s):
    return s.replace(' ', '_').replace('(', '').replace(')', '')

# Save MobileNetV2 classifiers
for name, r in results_mob.items():
    path = f"{SAVE_DIR}/mob_{safe_name(name)}.pkl"
    joblib.dump(r['clf'], path)
    print(f"Saved: {path}")

# Save ResNet50 classifiers
for name, r in results_res.items():
    path = f"{SAVE_DIR}/res_{safe_name(name)}.pkl"
    joblib.dump(r['clf'], path)
    print(f"Saved: {path}")

# Save shared assets
joblib.dump(le, f'{SAVE_DIR}/label_encoder.pkl')
mobilenet_extractor.save(f'{SAVE_DIR}/mobilenet_extractor.h5')
resnet_extractor.save(f'{SAVE_DIR}/resnet_extractor.h5')

with open(f'{SAVE_DIR}/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

print(f"\nAll models saved to: {SAVE_DIR}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 11 — Grad-CAM (Explainable AI) using MobileNetV2
#
# Grad-CAM computes the gradient of the most activated feature
# (chosen by the backbone) w.r.t. the last Conv2D layer's output.
# The resulting heatmap highlights regions that most influenced
# the network's feature extraction for this image.
# ════════════════════════════════════════════════════════════════

def build_gradcam_model(img_size=(128, 128)):
    """
    Build a two-output MobileNetV2 model:
      output[0] — last Conv2D feature maps  (spatial, for gradient computation)
      output[1] — GlobalAveragePooling vec  (same 1280-d as the feature extractor)
    """
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(*img_size, 3)
        # no pooling — keeps spatial dimensions
    )
    base.trainable = False

    # Identify the last Conv2D layer (MobileNetV2: typically 'Conv_1')
    last_conv_name = next(
        l.name for l in reversed(base.layers)
        if isinstance(l, tf.keras.layers.Conv2D)
    )
    print(f"Grad-CAM target layer : {last_conv_name}")

    # Attach a GAP on top of the base output (mirrors feature extractor)
    gap = tf.keras.layers.GlobalAveragePooling2D()(base.output)

    grad_model = tf.keras.Model(
        inputs=base.inputs,
        outputs=[base.get_layer(last_conv_name).output, gap]
    )
    return grad_model, last_conv_name


def compute_gradcam(img_array, grad_model):
    """
    Compute Grad-CAM heatmap for img_array.

    Gradient of the most-activated feature dimension w.r.t. the
    last conv layer is computed, pooled across channels, and used
    to weight each spatial feature map.

    Returns a normalised heatmap array in [0, 1].
    """
    with tf.GradientTape() as tape:
        conv_output, features = grad_model(img_array, training=False)
        # Target: the dimension with the highest activation
        pred_idx = int(tf.argmax(features[0]))
        score    = features[:, pred_idx]

    # Gradients of the score w.r.t. conv feature maps
    grads       = tape.gradient(score, conv_output)          # (1, h, w, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))     # (C,)

    conv_output = conv_output[0]                             # (h, w, C)
    heatmap     = conv_output @ pooled_grads[..., tf.newaxis]  # (h, w, 1)
    heatmap     = tf.squeeze(heatmap)
    heatmap     = tf.nn.relu(heatmap)
    heatmap     = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def show_gradcam(img_path, img_size=MOBILENET_IMG_SIZE, save_path=None):
    """
    Load an MRI image, run Grad-CAM, and display:
      - Original MRI
      - Grad-CAM heatmap
      - Heatmap overlaid on MRI
    """
    grad_model, _ = build_gradcam_model(img_size)

    # Preprocess image
    img = Image.open(img_path).convert('RGB').resize(img_size)
    arr = mobilenet_preprocess(
        np.expand_dims(np.array(img, dtype=np.float32), axis=0)
    )

    # Compute heatmap and resize to image dimensions
    heatmap        = compute_gradcam(arr, grad_model)
    heatmap_resized = cv2.resize(heatmap, img_size)
    heatmap_color   = cv2.applyColorMap(
        np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET
    )
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Blend heatmap with original
    orig_array = np.array(img)
    overlay    = np.clip(0.55 * orig_array + 0.45 * heatmap_color, 0, 255).astype(np.uint8)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('Grad-CAM Explainability — MobileNetV2', fontsize=14, fontweight='bold')

    axes[0].imshow(orig_array);             axes[0].set_title('Original MRI');         axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet'); axes[1].set_title('Grad-CAM Heatmap'); axes[1].axis('off')
    axes[2].imshow(overlay);               axes[2].set_title('Overlay');              axes[2].axis('off')

    # Colorbar for heatmap panel
    sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=0, vmax=1))
    plt.colorbar(sm, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    sp = save_path or f"{SAVE_DIR}/gradcam_output.png"
    plt.savefig(sp, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Grad-CAM saved to: {sp}")
    return heatmap


print("Grad-CAM functions defined. Call show_gradcam(img_path) to visualise.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 12 — Multi-Model Prediction
#
# For a new MRI image, runs ALL 6 models:
#   MobileNetV2 + {SVM, Random Forest, Gradient Boosting}
#   ResNet50    + {SVM, Random Forest, Gradient Boosting}
# Displays predictions side-by-side, then generates Grad-CAM.
# ════════════════════════════════════════════════════════════════

def predict_all_models(img_path):
    """
    Run a single MRI image through all trained classifiers
    for both backbones and display a comparative prediction table
    along with a Grad-CAM explanation.

    Parameters
    ----------
    img_path : str
        Path to the MRI image file (.jpg / .jpeg / .png).

    Returns
    -------
    all_results : dict
        Nested dict {backbone -> {classifier -> {'label', 'confidence'}}}.
    """
    print('=' * 60)
    print("  ALZHEIMER'S DETECTION — MULTI-MODEL PREDICTION")
    print('=' * 60)
    print(f"  Image : {img_path}\n")

    # ── Feature extraction for both backbones ──────────────────
    def get_features(path, extractor, preprocess_fn, img_size):
        img = Image.open(path).convert('RGB').resize(img_size)
        arr = preprocess_fn(
            np.expand_dims(np.array(img, dtype=np.float32), axis=0)
        )
        return extractor.predict(arr, verbose=0)

    feat_mob = get_features(img_path, mobilenet_extractor, mobilenet_preprocess, MOBILENET_IMG_SIZE)
    feat_res = get_features(img_path, resnet_extractor,    resnet_preprocess,    RESNET_IMG_SIZE)

    # ── Collect predictions from all 6 models ─────────────────
    all_results = {}

    for backbone, results, feat in [
        ('MobileNetV2', results_mob, feat_mob),
        ('ResNet50',    results_res, feat_res)
    ]:
        all_results[backbone] = {}
        print(f"  ── {backbone} ──────────────────────────")
        for clf_name, r in results.items():
            pred_enc   = r['clf'].predict(feat)[0]
            pred_label = le.inverse_transform([pred_enc])[0]

            # Confidence from probability estimate (if available)
            if hasattr(r['clf'], 'predict_proba'):
                proba      = r['clf'].predict_proba(feat)[0]
                confidence = proba[pred_enc] * 100
            else:
                confidence = None

            all_results[backbone][clf_name] = {
                'label':      pred_label,
                'confidence': confidence
            }

            conf_str = f"  ({confidence:.1f}% confidence)" if confidence is not None else ''
            print(f"    {clf_name:<22} →  {pred_label}{conf_str}")
        print()

    # ── Visualisation: image + two prediction tables ───────────
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle('Multi-Model Prediction — All Classifiers & Backbones',
                 fontsize=14, fontweight='bold')

    # Original image
    orig = np.array(Image.open(img_path).convert('RGB').resize(MOBILENET_IMG_SIZE))
    axes[0].imshow(orig)
    axes[0].set_title('Input MRI', fontweight='bold')
    axes[0].axis('off')

    def draw_table(ax, results_dict, backbone_label):
        ax.axis('off')
        cell_text  = []
        row_labels = []
        for clf_name, info in results_dict.items():
            row_labels.append(clf_name)
            conf_str = f"{info['confidence']:.1f}%" if info['confidence'] is not None else 'N/A'
            cell_text.append([info['label'], conf_str])

        tbl = ax.table(
            cellText=cell_text,
            rowLabels=row_labels,
            colLabels=['Prediction', 'Confidence'],
            cellLoc='center',
            loc='center',
            bbox=[0, 0.1, 1, 0.8]
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)
        # Header row styling
        for (row, col), cell in tbl.get_celld().items():
            if row == 0:
                cell.set_facecolor('#2c3e50')
                cell.set_text_props(color='white', fontweight='bold')
            elif col == -1:
                cell.set_facecolor('#ecf0f1')
        ax.set_title(f'{backbone_label} Classifiers', fontweight='bold', pad=12)

    draw_table(axes[1], all_results['MobileNetV2'], 'MobileNetV2')
    draw_table(axes[2], all_results['ResNet50'],    'ResNet50')

    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/prediction_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Grad-CAM explanation ───────────────────────────────────
    print("  Generating Grad-CAM heatmap (MobileNetV2)...")
    show_gradcam(img_path)

    return all_results


print("predict_all_models() defined. Run the next cell to test on a sample image.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Cell 13 — Demo: Run Multi-Model Prediction on a Sample Image
#
# Replace SAMPLE_IMG with any MRI image path to test your own.
# ════════════════════════════════════════════════════════════════

# Pick the first image from the first class in the test set
first_class = CLASS_NAMES[0]
first_img   = sorted(os.listdir(os.path.join(TEST_DIR, first_class)))[0]
SAMPLE_IMG  = os.path.join(TEST_DIR, first_class, first_img)

print(f"Sample image : {SAMPLE_IMG}")
print(f"True label   : {first_class}\n")

prediction_results = predict_all_models(SAMPLE_IMG)